In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
torch.manual_seed(2025)

### P2_P1 (Day 3) (5 points, non-coding task)

The high level idea of affine transformation in math is that for each column vector $x \in \mathbb{R}^N$, an affine transformation maps it to another column vector $y \in \mathbb{R}^M$ via

$$y = Wx + b$$

where $W \in \mathbb{R}^{M \times N}$ and $b \in \mathbb{R}^M$.

Let $W = \begin{bmatrix} 2 & -3 & 1 & 3 & -2 \\ 0 & 1 & 2 & 5 & -1 \\ 7 & -1 & -3 & 7 & 0 \end{bmatrix}$, $b = \begin{bmatrix} 1 \\ 0 \\ -1 \end{bmatrix}$, and $x = \begin{bmatrix} 1 \\ 2 \\ -3 \\ 1 \\ -2 \end{bmatrix}$.

1. **What is the value of $N$?**

2. **What is the value of $M$?**

3. **What is the value of $y$?**

#### Solution to P2_P1

*Write your answers here.*

1. **N =**

2. **M =**

3. **y =**

### P2_P3 (Day 3) (10 points, coding task)

In this part, you are asked to build an affine transformation module from scratch by using **NumPy**, NOT PyTorch or TensorFlow.

Define such a class as `My_Linear_NumPy`.

- **Attributes**
    - `in_features`: Number of input features
    - `out_features`: Number of output features
    - `weight`: This refers to matrix $W$ in Part 1. The shape is `(out_features, in_features)`.
    - `bias`: This refers to vector $b$ in Part 1. The shape is `(out_features,)`.
    - `random_seed`: The NumPy random seed number used to generate initial values of weight and bias.
- **Method `__init__`**:
    - To initialize an object in this class, you need to specify `in_features` and `out_features`.
    - You may initialize the object by specifying a value for `random_seed`. If it is not specified, then its default value is 42.
    - The initial values of weight and bias are random that follow standard normal distributions generated with the seed number attribute `random_seed`.
- **Method `forward`**:
    - Input `x`: numpy array with shape $(n_0, n_1, \dots, n_{d-1}, \text{in\_features})$ with an arbitrary dimension $d=1, 2, \dots$.
    - Output `y`: numpy array with shape $(n_0, n_1, \dots, n_{d-1}, \text{out\_features})$.
    - The affine transformation works in a way that given the first $d$ indices in `x` and `y`, it does affine transformation along the last axis of `x` and `y`.
    - **Do not use any loop in your code.**

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class My_Linear_NumPy:
    pass

""" END OF THIS PART """

In [ ]:
class My_Linear_NumPy:
    def __init__(self, in_features, out_features, random_seed=42):
        self.in_features = in_features
        self.out_features = out_features
        self.random_seed = random_seed
        
        # Initialize weight and bias
        np.random.seed(self.random_seed)
        self.weight = np.random.randn(self.out_features, self.in_features)
        self.bias = np.random.randn(self.out_features)
        
    def forward(self, x):
        """
        x shape: (..., in_features)
        weight shape: (out_features, in_features)
        bias shape: (out_features,)
        """
        return x @ self.weight.T + self.bias # Looks at the last dimension of x and the second to last dimension of weight, multiplies them together, and sums over that dimension. Then adds the bias.

# Verification with example from Part 1
model = My_Linear_NumPy(in_features=5, out_features=3, random_seed=None)
model.weight = np.array([[2, -3, 1, 3, -2],
                         [0, 1, 2, 5, -1],
                         [7, -1, -3, 7, 0]])
model.bias = np.array([1, 0, -1])

x_np = np.array([1, 2, -3, 1, -2])
y_np = model.forward(x_np)
print(f"y (NumPy) = {y_np}")

If `x` has shape **(2, 3, 4)** and `weight` has shape **(5, 4)**, the operation works perfectly because of how NumPy handles **broadcasting** and the **matmul (`@`)** operator.

Here is the step-by-step breakdown of the shapes:

### 1. The Matrix Multiplication: `x @ self.weight.T`
* **`x` shape:** `(2, 3, 4)` 
    * Think of this as 2 matrices, each of size $3 \times 4$.
* **`self.weight.T` shape:** `(4, 5)` 
    * This is a single matrix of size $4 \times 5$.

When you use `@`, NumPy keeps the leading "batch" dimensions (the **2** and the **3**) and performs the dot product on the last dimension of `x` (the **4**) and the first dimension of `weight.T` (also **4**).

* **Calculation:** `(2, 3, 4) @ (4, 5)` $\rightarrow$ **`(2, 3, 5)`**



---

### 2. Adding the Bias: `+ self.bias`
* **Intermediate result shape:** `(2, 3, 5)`
* **`self.bias` shape:** `(5,)`

NumPy uses **broadcasting** here. It looks at the last dimension of the intermediate result (which is **5**) and sees it matches the bias shape (**5**). It then "stretches" the bias vector across the other dimensions (the 2 and the 3) so that the same bias vector is added to every single row in your batch.

* **Calculation:** `(2, 3, 5) + (5,)` $\rightarrow$ **`(2, 3, 5)`**

---

### Final Result Summary
The output `y` will have a shape of **`(2, 3, 5)`**.

| Variable | Shape | Role |
| :--- | :--- | :--- |
| `x` | `(2, 3, 4)` | Input (Batch=2, Sequence=3, Features=4) |
| `weight` | `(5, 4)` | Weights (Out=5, In=4) |
| `weight.T` | `(4, 5)` | Transposed for multiplication |
| **Result** | **`(2, 3, 5)`** | Output (Batch=2, Sequence=3, New Features=5) |

### Why this is powerful
This setup allows your class to handle any number of "outer" dimensions. Whether you pass in a single vector `(4,)`, a batch of vectors `(32, 4)`, or a video-like structure `(Batch, Frames, Height, Width, 4)`, the `@` operator will always apply the transformation to that last dimension of **4** and return the new feature dimension of **5**.

Does this make the "batch processing" logic clearer for your assignment?